In [1]:
import ROOT
import numpy as np
import json
import copy
import os

In [2]:
def sideband_csv_fast(index, mode, spread=11, samplefreq=50000, verbose=False):
    """
    Optimized version of sideband_csv.

    Parameters
    ----------
    index : dict
        Prepared JSON-style index with structure:
        index[angle][position][hist_name]

    mode : int
        Mode 1 uses sidebands [-4, -2, 0, 2, 4]
        Mode 2 uses sidebands [-3, -1, 1, 3]

    spread : int, optional
        Width parameter in units of samplefreq.
        If even, it is increased by 1.

    samplefreq : float, optional
        Sample frequency used to define sideband windows.

    verbose : bool, optional
        If True, prints progress.

    Returns
    -------
    index_out : dict
        Cleaned index containing only PSD entries and JSON entries.
        PSD entries are replaced by sideband power/center data.
    """

    # Takes input options 
    if spread % 2 != 1:
        spread += 1

    if mode == 1:
        bands = np.array([-2, 0, 2])
    elif mode == 2:
        bands = np.array([-1, 1])
    else:
        raise ValueError("mode must be 1 for a TE0_11 mode or 2 for a TE0_12 mode")
    
    #prep final output
    index_out = {}

    jsn_name = "LocustEventProperties.json"

    #begin itteration
    for angle in index:
        index_out[angle] = {}
        for position in index[angle]:
            if verbose:
                print(f"Processing angle={angle}, position={position}", flush=True)

            index_out[angle][position] = {}

            jsn_info = index[angle][position][jsn_name]
            jsn_filepath = jsn_info["filepath"]

            with open(jsn_filepath, "r") as f:
                jsn = json.load(f)
            
            avgax = jsn["0"]["0"]["avg-axial-frequency"]
            central = jsn["0"]["0"]["output-avg-frequency"]

            band_ranges = []

            for band in bands:
                band_loc = central + (band * avgax)
                band_range = [
                    band_loc - (spread * samplefreq),
                    band_loc + (spread * samplefreq),
                ]
                band_ranges.append(band_range)

            # Finding all filepaths so it only needs to be loaded once
            psd_by_filepath = {}

            for hist_name, hist_info in index[angle][position].items():
                if "PSD" not in hist_name:
                    continue

                filepath = hist_info["filepath"]

                if filepath not in psd_by_filepath:
                    psd_by_filepath[filepath] = []

                psd_by_filepath[filepath].append(hist_name)
            
            #loading file paths to process hists
            for filepath, hist_names in psd_by_filepath.items():

                root_file = ROOT.TFile.Open(filepath, "READ")

                if not root_file or root_file.IsZombie():
                    raise RuntimeError(f"Could not open ROOT file: {filepath}")
                
                try:
                    for hist_name in hist_names:
                        hist = root_file.Get(hist_name)

                        if not hist:
                            raise RuntimeError(
                                f"Missing histogram {hist_name} in file {filepath}"
                            )
                        xaxis = hist.GetXaxis()
                        nbins = hist.GetNbinsX()

                        sidebands = []

                        for band, (low, high) in zip(bands, band_ranges):

                            bin_low = xaxis.FindBin(low)
                            bin_high = xaxis.FindBin(high)

                            bin_low = max(1, bin_low)
                            bin_high = min(nbins, bin_high)

                            counts = []
                            centers = []

                            for i in range(bin_low, bin_high + 1):
                                center = hist.GetBinCenter(i)
                                
                                if center >= low and center <= high:
                                        centers.append(center)
                                        counts.append(hist.GetBinContent(i))
                            if len(counts) == 0:
                                power = 0.0
                                center = 0.0

                            else:
                                max_i = int(np.argmax(counts))
                                power = float(counts[max_i])
                                center = float(centers[max_i])
                            if power == 0:
                                loc = np.inf
                            else:
                                loc = int(band)

                            sidebands.append([power, center,loc])
                            index_out[angle][position][hist_name] = {
                                "side_bands_powercenter": sidebands
                            }

                finally:
                    root_file.Close()

    return index_out


In [3]:
def Peaks(index,numbands=5,spread=4, verbose=False):
    index_out = {}
    jsn_name = "LocustEventProperties.json"
    for angle in index:
        index_out[angle] = {}
        for position in index[angle]:
            if verbose:
                print(f"Processing angle={angle}, position={position}", flush=True)   
                
            index_out[angle][position] = {}

            psd_by_filepath = {}
            
            for hist_name, hist_info in index[angle][position].items():
                if "PSD" not in hist_name:
                    continue

                filepath = hist_info["filepath"]

                if filepath not in psd_by_filepath:
                    psd_by_filepath[filepath] = []

                psd_by_filepath[filepath].append(hist_name)
            for filepath, hist_names in psd_by_filepath.items():

                root_file = ROOT.TFile.Open(filepath, "READ")

                if not root_file or root_file.IsZombie():
                    raise RuntimeError(f"Could not open ROOT file: {filepath}")
                try:
                    for hist_name in hist_names:
                        hist = root_file.Get(hist_name)

                        if not hist:
                            raise RuntimeError(
                                f"Missing histogram {hist_name} in file {filepath}"
                            )
                        xaxis = hist.GetXaxis()
                        nbins = hist.GetNbinsX()

                        sidebands = []
                        positions=[]

                        counts = np.array([
                            hist.GetBinContent(i)
                            for i in range(1, nbins + 1)
                        ])
                        centers=np.array([
                            hist.GetBinCenter(i)
                            for i in range(1, nbins + 1)
                        ])
                        bands_found=0 
                        while bands_found<numbands: 
                            mx=np.max(counts) 
                            pos=int(np.where(counts==mx)[0][0]) 
                            center=float(centers[pos]) 
                            power=float(counts[pos]) 
                            match_index = next( (i for i, old_pos in enumerate(positions) if abs(pos - old_pos) <= spread), None ) 
                            if match_index is not None: 
                                sidebands[match_index][0] +=power 
                            else:
                                sidebands.append([power,center])
                                positions.append(pos) 
                                bands_found+=1
                            counts[pos] = counts[pos]-1

                        sidebands = np.array(sidebands)

                        sidebands_sorted = sidebands[np.argsort(sidebands[:, 1])]

                        index_out[angle][position][hist_name] = {
                            "side_bands_powercenter": sidebands_sorted.tolist()
                        }
                finally:
                    root_file.Close()

    return index_out

                            
                    

In [4]:
base_path1="Band_class_training data/TE_011/Results"
index_path1 = os.path.join(base_path1, "Te011.json")
with open(index_path1, "r") as f:
    TE011_Index = json.load(f)

In [5]:
base_path2 = "Band_class_training data/TE_012/Results"
index_path2 = os.path.join(base_path2, "Te012.json")
with open(index_path2, "r") as f:
    TE012_Index = json.load(f)

In [6]:
def save_index(index,position):
    with open(position, "w") as f:
        json.dump(index, f, indent=4)

In [7]:
sidebands1=sideband_csv_fast(TE011_Index,1)
sidebands2=sideband_csv_fast(TE012_Index,2)

In [8]:
maxbands1=Peaks(TE011_Index)
maxbands2=Peaks(TE012_Index)

In [9]:
if __name__ == "__main__":
    output_file = os.path.join(base_path1, "Te011_sidebands.json")
    save_index(sidebands1,output_file)

In [10]:
if __name__ == "__main__":
    output_file = os.path.join(base_path2, "Te012_sidebands.json")
    save_index(sidebands2,output_file)

In [11]:
if __name__ == "__main__":
    output_file = os.path.join(base_path1, "Te011_MaxBands.json")
    save_index(maxbands1,output_file)

In [12]:
if __name__ == "__main__":
    output_file = os.path.join(base_path2, "Te012_MaxBands.json")
    save_index(maxbands2,output_file)

In [13]:
maxbands1['Angle88.0000000']['Pos0.0030000']['histPSD_12_0']

{'side_bands_powercenter': [[4.936720511028581e-13, 112310668.9453125],
  [3.335056731705538e-12, 125052001.953125],
  [6.6168037653110804e-12, 137842529.296875],
  [2.846431131427778e-12, 150583862.3046875],
  [2.26005902970083e-14, 163374389.6484375]]}

In [14]:
sidebands1['Angle88.0000000']['Pos0.0030000']['histPSD_12_0']

{'side_bands_powercenter': [[5.17661830308968e-12, 137842529.296875, -2],
  [2.4468695844067624e-12, 150583862.3046875, 0],
  [2.26005902970083e-14, 163374389.6484375, 2]]}